# Train and compare BDT feature sets across pT bins

Edit `multiModelsBDT_config.json` to choose the named feature lists and `PT_BINS`. Every feature-set/bin combination is trained with the same split seed.


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import auc, roc_curve
from hipe4ml import plot_utils
import singleModelBDT

CONFIG_PATH = Path('multiModelsBDT_config.json')
with CONFIG_PATH.open() as config_file:
    config = json.load(config_file)

FEATURE_SETS = config['FEATURE_SETS']
PT_BINS = [tuple(pt_bin) for pt_bin in config['PT_BINS']]
OUTPUT_ROOT = Path(config['OUT_DIR'])
TRAINING = config.get('TRAINING', {})
CLASS_LABELS = ['background', 'prompt', 'nonprompt']
if not FEATURE_SETS or not PT_BINS:
    raise ValueError('FEATURE_SETS and PT_BINS must both be non-empty')


## Train every configured combination


In [ ]:
from importlib import reload
reload(singleModelBDT)
training_results = {}
for model_name, features in FEATURE_SETS.items():
    for pt_bin in PT_BINS:
        pt_tag = f"pt{pt_bin[0]}_{pt_bin[1]}"
        run_dir = OUTPUT_ROOT / model_name / pt_tag
        print(f'Training {model_name} for pT {pt_bin[0]}–{pt_bin[1]} in {run_dir}')
        training_results[(model_name, pt_bin)] = singleModelBDT.runSingleModel(
            features=features,
            pt_bin=pt_bin,
            model_name=model_name,
            out_dir=run_dir,
            data_file=config['DATA_FILE'],
            promptmc_file=config['PROMPTMC_FILE'],
            nonpromptmc_file=config['NONPROMPTMC_FILE'],
            n_estimators=tuple(TRAINING.get('N_ESTIMATORS', [20, 1000])),
            max_depth=tuple(TRAINING.get('MAX_DEPTH', [2, 6])),
            learning_rate=tuple(TRAINING.get('LEARNING_RATE', [0.01, 0.1])),
            n_jobs=TRAINING.get('N_JOBS', 10),
            timeout=TRAINING.get('TIMEOUT', 60),
            n_trials=TRAINING.get('N_TRIALS', 100),
            random_state=TRAINING.get('RANDOM_STATE', 42),
            plotFigures=False,
        )

print(f'Trained {len(training_results)} model/bin configurations')


## Compare ROC curves


In [ ]:
comparison_results = {}
for (model_name, pt_bin), result in training_results.items():
    with np.load(result['comparison_file']) as saved:
        comparison_results[(model_name, pt_bin)] = {
            'y_test': saved['y_test'].copy(),
            'y_score': saved['y_score_test'].copy(),
        }

comparison_dir = OUTPUT_ROOT / 'comparison'
comparison_dir.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
for class_id, ax in enumerate(axes):
    for (model_name, pt_bin), result in comparison_results.items():
        y_binary = (result['y_test'] == class_id).astype(int)
        fpr, tpr, _ = roc_curve(y_binary, result['y_score'][:, class_id])
        curve_auc = auc(fpr, tpr)
        label = f"{model_name}, pT {pt_bin[0]}–{pt_bin[1]} (AUC={curve_auc:.3f})"
        ax.plot(fpr, tpr, label=label)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set(title=f'{CLASS_LABELS[class_id]} one-vs-rest', xlabel='False positive rate', ylabel='True positive rate')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.3)
    ax.legend(fontsize='x-small')
fig.tight_layout()
fig.savefig(comparison_dir / 'model_comparison_roc.pdf', bbox_inches='tight')
plt.show()
plt.close(fig)


## Compare pT efficiencies

Each figure uses the same pT and efficiency axes. Solid curves show prompt efficiency; dashed curves show nonprompt efficiency. Colors identify feature sets.


In [ ]:
NnonpCuts = 6
efficiency_cuts = np.round(np.linspace(0, 0.95, NnonpCuts), 2)
for pt_bin in PT_BINS:
    for efficiency_index, nonprompt_cut in enumerate(efficiency_cuts):
        fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)
        for model_name in FEATURE_SETS:
            result = training_results[(model_name, pt_bin)]
            with np.load(result['comparison_file']) as saved:
                centers = saved[f'pt_centers_{efficiency_index}']
                prompt_eff = saved[f'prompt_eff_{efficiency_index}']
                prompt_err = saved[f'prompt_err_{efficiency_index}']
                nonprompt_eff = saved[f'nonprompt_eff_{efficiency_index}']
                nonprompt_err = saved[f'nonprompt_err_{efficiency_index}']
            axes[0].errorbar(centers, prompt_eff, yerr=prompt_err, marker='o', capsize=2, label=model_name)
            axes[1].errorbar(centers, nonprompt_eff, yerr=nonprompt_err, marker='o', capsize=2, label=model_name)
        for ax, title in zip(axes, ('Prompt efficiency', 'Nonprompt efficiency')):
            ax.set(title=title, xlabel=r'$p_{\mathrm{T}}^{\mathrm{J}/\psi}$ (GeV/$c$)', ylabel='Efficiency')
            ax.set_xlim(0, 10)
            ax.set_ylim(0, 1.05)
            ax.grid(alpha=0.3)
            ax.legend(title='Feature set', fontsize='small')
        fig.suptitle(f'pT efficiency comparison, pT model bin {pt_bin[0]}–{pt_bin[1]}, nonprompt score cut {nonprompt_cut:.2f}')
        fig.tight_layout()
        tag = f'pt{pt_bin[0]}_{pt_bin[1]}'
        fig.savefig(comparison_dir / f'efficiency_comparison_{tag}_nonpromptCut{nonprompt_cut:.2f}.pdf', bbox_inches='tight')
        plt.show()
        plt.close(fig)
